# Encoder-Decoder Transformer — Machine Translation

The previous lesson built an **encoder-only** transformer (like BERT) that
reads a whole sequence and produces a single classification output.

Translation requires a different shape: given a source sentence in one language,
generate a target sentence in another. This is a **sequence-to-sequence** (seq2seq)
problem, and it calls for an **encoder-decoder** transformer.

**Architecture:**
```
Source tokens  (B, S)
  → Encoder Embedding + PE     (B, S, d_model)
  → N × EncoderLayer           (B, S, d_model)
  → memory                     (B, S, d_model)

Target tokens  (B, T)          ← teacher-forced during training
  → Decoder Embedding + PE     (B, T, d_model)
  → N × DecoderLayer:
       Masked Self-Attention   (B, T, d_model)   ← causal: can't see future
       Cross-Attention         (B, T, d_model)   ← Q from tgt, K/V from memory
       Feed-Forward            (B, T, d_model)
  → Linear → logits            (B, T, tgt_vocab)
```

> **Shape notation:** `B` = batch size, `S` = source length, `T` = target length,
> `d_model` = embedding/hidden dimension.

Dataset: English–French sentence pairs from the NLTK `comtrans` corpus
(Canadian Hansard proceedings, ~900 pairs).  The goal is to build intuition
for the encoder-decoder architecture — real translation uses millions of pairs
or a fine-tuned model like T5/mBART.

> 📖 [Vaswani et al. — Attention Is All You Need (2017)](https://arxiv.org/abs/1706.03762)  
> 📖 [Alammar — The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)

## Step 1: Imports

In [ ]:
import copy
import math
import random
from collections import Counter

import nltk
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

device = torch.device('mps' if torch.backends.mps.is_available()
                       else 'cuda' if torch.cuda.is_available()
                       else 'cpu')


## Key Concept: Encoder-Decoder Architecture

The encoder reads the entire source sentence and produces a sequence of
contextualised representations — one vector per source token.
This sequence is called **memory**.

The decoder then generates the target sentence **one token at a time**.
At each step it has access to:
1. The tokens it has already generated (via **masked self-attention** — causal so
   it cannot peek at future tokens).
2. The full encoder output (via **cross-attention** — Q comes from the decoder,
   K and V come from memory).

**Why two separate attention mechanisms in the decoder?**
- Masked self-attention builds a representation of "what I've written so far."
- Cross-attention asks "given what I've written, which source words should I
  focus on next?"

**Cross-attention is the bridge** between the two languages. It is the only
place where source-language information flows into the decoder.

```
Cross-Attention:
  Q  ←  decoder hidden state   (B, T, d_model)
  K  ←  encoder memory         (B, S, d_model)
  V  ←  encoder memory         (B, S, d_model)
  output shape: (B, T, d_model)   — each decoder position attends over all S source positions
```

## Step 2: Load and Preprocess Translation Data

In [ ]:
nltk.download('comtrans', quiet=True)
from nltk.corpus import comtrans

PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

MAX_LEN   = 16    # max tokens per sentence (excluding SOS/EOS)
MAX_VOCAB = 2000  # vocab size per language

random.seed(42)
# comtrans ships three alignment files (de-en, de-fr, en-fr). Calling
# aligned_sents() with no argument concatenates ALL of them, which would
# train one model on three different language pairs at once. Name the file.
pairs = comtrans.aligned_sents('alignment-en-fr.txt')

# Filter to short sentences so training is fast
en_sents, fr_sents = [], []
for p in pairs:
    src = [w.lower() for w in p.words]
    tgt = [w.lower() for w in p.mots]
    if len(src) <= MAX_LEN and len(tgt) <= MAX_LEN:
        en_sents.append(src)
        fr_sents.append(tgt)

print(f'Sentence pairs after filtering: {len(en_sents)}')
print('English example:', en_sents[0])
print('French  example:', fr_sents[0])

def build_vocab(sentences, max_vocab):
    vocab = {PAD: PAD_IDX, SOS: SOS_IDX, EOS: EOS_IDX, UNK: UNK_IDX}
    freq  = Counter(w for sent in sentences for w in sent)
    for w, _ in freq.most_common(max_vocab - len(vocab)):
        vocab[w] = len(vocab)
    return vocab

en_vocab = build_vocab(en_sents, MAX_VOCAB)
fr_vocab = build_vocab(fr_sents, MAX_VOCAB)
print(f'English vocab: {len(en_vocab)}  French vocab: {len(fr_vocab)}')

def encode_sent(words, vocab, max_len):
    """Encode to [SOS, w1, w2, ..., EOS, PAD, PAD, ...]."""
    ids = [SOS_IDX] + [vocab.get(w, UNK_IDX) for w in words[:max_len]] + [EOS_IDX]
    ids += [PAD_IDX] * (max_len + 2 - len(ids))
    return ids

src_data = torch.tensor([encode_sent(s, en_vocab, MAX_LEN) for s in en_sents], dtype=torch.long)
tgt_data = torch.tensor([encode_sent(s, fr_vocab, MAX_LEN) for s in fr_sents], dtype=torch.long)
print(f'src_data: {src_data.shape}   tgt_data: {tgt_data.shape}')  # (N, MAX_LEN+2)

## Step 3: DataLoaders

In [ ]:
N     = len(src_data)
split = int(0.85 * N)

train_ds = TensorDataset(src_data[:split], tgt_data[:split])
val_ds   = TensorDataset(src_data[split:], tgt_data[split:])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)
print(f'Train: {len(train_ds)} pairs  Val: {len(val_ds)} pairs')

## Key Concept: Causal Masking in the Decoder

The decoder uses **two** masking strategies:

### 1 — Causal mask (autoregressive mask)
During training, the decoder receives the entire target sequence at once
(teacher forcing — explained later).  To prevent position `t` from attending
to positions `> t`, we apply an upper-triangular mask:

```
Position:   0  1  2  3
        0 [ .  ✗  ✗  ✗ ]   position 0 can only see itself
        1 [ .  .  ✗  ✗ ]   position 1 can see 0 and 1
        2 [ .  .  .  ✗ ]   position 2 can see 0, 1, 2
        3 [ .  .  .  . ]   position 3 can see everything
```

In `scaled_dot_product_attention`, we fill masked positions with `-inf`
before softmax — they become 0 weight after softmax.

### 2 — Padding mask
PAD tokens (index 0) are inserted to make all sequences the same length.
We don't want attention to treat them as meaningful; we mask them out
in the key dimension so no real token attends to a PAD position.

**Why is the causal mask critical for generation?**
At inference we generate token by token.  During training we simulate this
with the full target sequence.  Without the causal mask, position `t` would
see the answer it's supposed to predict — the task becomes trivial and the
model learns nothing useful.

## Step 4: Masking Utilities

In [ ]:
def make_causal_mask(L, device):
    """
    Returns a (L, L) boolean mask where True means "masked out".
    True in position (i, j) means token i should NOT attend to token j.
    The upper triangle (j > i) is True — future positions are blocked.
    """
    # TODO: use torch.triu to create the upper-triangular boolean mask
    # torch.triu(M, diagonal=1) keeps the triangle strictly above the diagonal
    return ...


# Quick test
m = make_causal_mask(4, 'cpu')
print('Causal mask (4x4):\n', m)
# Expected: each row has True only in positions to the right of the diagonal
# Row 0: [F T T T]  row 1: [F F T T]  row 2: [F F F T]  row 3: [F F F F]

### Concept Check: Masking

**Q1.** The causal mask has shape `(T, T)` but attention scores have shape
`(B, H, T, T)`. How does PyTorch broadcasting handle this?

**Q2.** During *inference* we generate one token at a time, so we never have
the full target sequence available.  Does the causal mask matter at inference
time?  Why or why not?

**Q3.** If you forgot the causal mask during training but used it at inference,
describe the training/inference mismatch and predict its effect on output quality.

In [ ]:
# Q1:
# Q2:
# Q3:

## Step 5: Shared Components (from Lesson 07)

These are identical to Lesson 07 — reproduced here so the notebook is standalone.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, H, L_q, d_k)  K: (B, H, L_k, d_k)  V: (B, H, L_k, d_v)
    mask: broadcastable boolean tensor — True means mask out (set to -inf)
    Returns: output (B, H, L_q, d_v), weights (B, H, L_q, L_k)
    """
    d_k     = Q.size(-1)
    scores  = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B,H,L_q,L_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.num_heads, self.d_k).transpose(1, 2)  # (B,H,L,d_k)

    def forward(self, Q, K, V, mask=None):
        B  = Q.size(0)
        Qh = self.split_heads(self.W_q(Q))  # (B,H,L_q,d_k)
        Kh = self.split_heads(self.W_k(K))  # (B,H,L_k,d_k)
        Vh = self.split_heads(self.W_v(V))  # (B,H,L_k,d_k)
        out, weights = scaled_dot_product_attention(Qh, Kh, Vh, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)
        return self.W_o(out), weights


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                        * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):  # x: (B, L, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


print('Shared components defined.')

## Key Concept: Cross-Attention

In the encoder, attention is **self-attention**: Q, K, and V all come from the
same sequence.  Each position asks *"which other positions in my sentence
are relevant to me?"*

In the decoder, the second sub-layer uses **cross-attention**:
- Q comes from the **decoder** hidden states (T positions)
- K and V come from the **encoder memory** (S positions)

Each decoder position asks *"given what I'm generating, which encoder
positions (source words) should I attend to?"*

The attention weight matrix has shape `(B, H, T, S)` — `T` decoder
positions, `S` source positions.  When you visualise it you can literally
see which source words the model "looks at" when generating each output word.

```
Cross-attention score[i, j] = relevance of source position j
                               when generating target position i
```

**No causal mask is used in cross-attention** — the decoder is allowed to
attend to any source position freely.  The causal mask only applies to the
decoder's self-attention to prevent it from peeking at future *target* tokens.

## Step 6: Transformer Decoder Layer

In [ ]:
class TransformerDecoderLayer(nn.Module):
    """
    One decoder layer:
      Sub-layer 1: Masked self-attention (causal)         — tgt attends to tgt
      Sub-layer 2: Cross-attention                        — tgt attends to memory
      Sub-layer 3: Position-wise Feed-Forward
    Each sub-layer uses a residual connection and LayerNorm.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None):
        """
        tgt:    (B, T, d_model)   — decoder input embeddings
        memory: (B, S, d_model)   — encoder output
        tgt_mask: (T, T) boolean  — causal mask for self-attention
        Returns: (B, T, d_model)
        """
        # ── Sub-layer 1: Masked self-attention ──────────────────────────────
        # Self-attention: Q = K = V = tgt, with causal tgt_mask
        # TODO: call self.self_attn(tgt, tgt, tgt, mask=tgt_mask), unpack result
        sa_out, _ = ...
        # TODO: residual + dropout + LayerNorm
        tgt = ...

        # ── Sub-layer 2: Cross-attention ─────────────────────────────────────
        # Q comes from the decoder (tgt), K and V come from the encoder (memory)
        # No mask — decoder can attend freely to all source positions
        # TODO: call self.cross_attn(tgt, memory, memory), unpack result
        ca_out, _ = ...
        # TODO: residual + dropout + LayerNorm
        tgt = ...

        # ── Sub-layer 3: Feed-Forward ────────────────────────────────────────
        # TODO: apply self.ff, then residual + dropout + LayerNorm
        tgt = ...

        return tgt


# Shape test
B, S, T, d = 2, 10, 8, 128
_mem = torch.randn(B, S, d)
_tgt = torch.randn(B, T, d)
_mask = make_causal_mask(T, 'cpu')
_dec = TransformerDecoderLayer(d_model=d, num_heads=4, d_ff=256)
_out = _dec(_tgt, _mem, tgt_mask=_mask)
print('DecoderLayer output:', _out.shape)  # expect (2, 8, 128)

### Concept Check: Decoder Layer

**Q1.** The decoder layer has three sub-layers while the encoder has two.
Which sub-layer is new, and what information does it introduce that the
encoder's self-attention cannot provide?

**Q2.** In cross-attention, Q has shape `(B, H, T, d_k)` and K has shape
`(B, H, S, d_k)`.  What shape does the attention weight matrix have?  What
does a high weight at position `[b, h, t, s]` mean in plain language?

**Q3.** During beam search (a more sophisticated decoding strategy), we
keep the top-`k` candidate sequences at each step.  The encoder is run
once; the decoder is run once per step.  Why is it efficient to run the
encoder only once even for beam search with `k=5`?

In [ ]:
# Q1:
# Q2:
# Q3:

## Step 7: Transformer Encoder Layer (from Lesson 07)

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):  # x: (B, S, d_model)
        attn_out, _ = self.attn(x, x, x, mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x

print('EncoderLayer defined.')

## Step 8: Full Seq2Seq Transformer

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 d_model=128, num_heads=4, num_layers=2,
                 d_ff=256, max_len=32, dropout=0.1):
        super().__init__()
        # Separate embeddings and positional encodings for source and target
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.src_pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.tgt_pos_enc   = PositionalEncoding(d_model, max_len, dropout)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def encode(self, src):
        """
        src: (B, S) token indices
        Returns memory: (B, S, d_model)
        """
        # TODO: embed src + add positional encoding
        x = ...
        # TODO: pass x through each layer in self.encoder_layers
        for layer in self.encoder_layers:
            x = ...
        return x  # memory

    def decode(self, tgt, memory):
        """
        tgt:    (B, T) token indices
        memory: (B, S, d_model)
        Returns: (B, T, d_model)
        """
        T = tgt.size(1)
        # TODO: create causal mask for the target sequence length T
        tgt_mask = ...
        # TODO: embed tgt + add positional encoding
        x = ...
        # TODO: pass x through each decoder layer (pass tgt_mask)
        for layer in self.decoder_layers:
            x = ...
        return x

    def forward(self, src, tgt):
        """
        src: (B, S)  tgt: (B, T)
        Returns logits: (B, T, tgt_vocab_size)
        """
        # TODO: encode, then decode, then project to vocab logits
        memory  = ...
        decoded = ...
        return ...


model = Seq2SeqTransformer(
    src_vocab_size=len(en_vocab),
    tgt_vocab_size=len(fr_vocab),
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Key Concept: Teacher Forcing

At inference the decoder generates tokens **autoregressively**: it produces
token `t` and feeds it back as input for step `t+1`.  Errors compound —
a wrong token at step 3 misleads step 4, and so on.

Training with the decoder's own (possibly wrong) outputs from the start
would be slow and unstable.  **Teacher forcing** sidesteps this by feeding
the *ground-truth* target token at every step, regardless of what the model
predicted:

```
Target sequence:  <SOS> le   chat  dort  <EOS>
Decoder input:    <SOS> le   chat  dort        ← tgt[:, :-1]  (all but last)
Decoder output:   le    chat dort  <EOS>       ← tgt[:, 1:]   (all but first)

At each position t, the model sees the correct token from position t-1
and must predict the correct token at position t.
```

**Effect:** The loss is computed over all `T-1` positions simultaneously in
one forward pass, making training much faster.

**Trade-off:** The distribution of inputs seen during training (ground-truth
tokens) differs from inference (model's own predictions) — this is called
**exposure bias**.  Scheduled sampling gradually switches from teacher-forced
to model-predicted inputs during training to reduce the gap.

## Step 9: Training Loop

In [ ]:
TGT_VOCAB_SIZE = len(fr_vocab)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)  # ignore PAD in loss
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

train_losses, val_losses = [], []

# This dataset is small (~8k pairs), so the model starts overfitting well
# before epoch 30: train loss keeps falling while val loss turns back up.
# Keep a copy of the weights from the best validation epoch and restore it
# at the end, so we evaluate the model at its best, not its last.
best_val, best_state, best_epoch = float('inf'), None, 0

for epoch in range(30):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total = 0
    for src_b, tgt_b in train_loader:
        src_b, tgt_b = src_b.to(device), tgt_b.to(device)

        # Teacher forcing:
        #   decoder input  = tgt_b[:, :-1]  →  <SOS> w1 w2 ... w_{T-1}
        #   decoder target = tgt_b[:, 1:]   →  w1 w2 ... w_{T-1} <EOS>
        logits = model(src_b, tgt_b[:, :-1])   # (B, T-1, tgt_vocab)

        # Flatten for cross-entropy: (B*(T-1), vocab) vs (B*(T-1),)
        loss = criterion(
            logits.reshape(-1, TGT_VOCAB_SIZE),
            tgt_b[:, 1:].reshape(-1),
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # stability
        optimizer.step()
        total += loss.item()

    avg_train = total / len(train_loader)
    train_losses.append(avg_train)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    vtotal = 0
    with torch.no_grad():
        for src_b, tgt_b in val_loader:
            src_b, tgt_b = src_b.to(device), tgt_b.to(device)
            logits = model(src_b, tgt_b[:, :-1])
            vtotal += criterion(logits.reshape(-1, TGT_VOCAB_SIZE),
                                tgt_b[:, 1:].reshape(-1)).item()
    avg_val = vtotal / len(val_loader)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    if avg_val < best_val:
        best_val, best_epoch = avg_val, epoch + 1
        best_state = copy.deepcopy(model.state_dict())

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d} | train {avg_train:.3f} | val {avg_val:.3f}')

model.load_state_dict(best_state)
print(f'\nRestored weights from epoch {best_epoch} (val {best_val:.3f}).')

## Key Concept: Autoregressive Decoding

At inference, ground-truth targets are not available.  The decoder must
generate each token from scratch:

```
Step 0:  input = [<SOS>]                  → predict w1
Step 1:  input = [<SOS>, w1]              → predict w2
Step 2:  input = [<SOS>, w1, w2]          → predict w3
...                                       stop when w_t = <EOS>
```

This is called **greedy decoding** — at each step, pick the single most
probable next token.  It is fast but not optimal.

**Beam search** keeps the top-`k` partial sequences at each step and picks
the globally highest-probability complete sequence.  It often produces better
translations but requires `k` times more computation.

**Why can't we run the full sequence in parallel at inference?**  
Each token depends on all previous tokens.  We don't know `w2` until we've
committed to `w1`, so decoding is inherently sequential — unlike training
with teacher forcing, which parallelises over the full known target.

## Step 10: Greedy Decode

In [ ]:
idx_to_fr = {v: k for k, v in fr_vocab.items()}

def greedy_decode(model, src_words, max_len=MAX_LEN + 2):
    """
    Translate a list of English word strings to French using greedy decoding.
    Returns a list of French word strings (without <SOS>/<EOS>).
    """
    model.eval()
    with torch.no_grad():
        # Encode source sentence
        src_ids = [en_vocab.get(w.lower(), UNK_IDX) for w in src_words]
        src_ids = [SOS_IDX] + src_ids[:MAX_LEN] + [EOS_IDX]
        src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)  # (1, S)

        # TODO: call model.encode(src_tensor) to get memory
        memory = ...

        # Autoregressive decoding — start with <SOS>
        tgt_ids = [SOS_IDX]
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long, device=device)  # (1, t)

            # TODO: call model.decode(tgt_tensor, memory) → (1, t, d_model)
            dec_out = ...

            # TODO: project to vocab with model.fc_out, take the last position,
            #       then take the argmax to get the next token index
            next_token = ...

            tgt_ids.append(next_token)
            if next_token == EOS_IDX:
                break

    # Strip <SOS> and <EOS> tokens and convert to words
    return [idx_to_fr.get(i, UNK) for i in tgt_ids[1:]
            if i not in (SOS_IDX, EOS_IDX, PAD_IDX)]


# Quick test — should produce some French words
test_src = en_sents[0]
print('EN:', ' '.join(test_src))
print('FR (ref):', ' '.join(fr_sents[0]))
print('FR (pred):', ' '.join(greedy_decode(model, test_src)))

## Step 11: Evaluate — Loss Curves and Sample Translations

In [ ]:
if HAS_MPL:
    plt.figure(figsize=(8, 3.5))
    plt.plot(train_losses, label='train', color='#a29bfe', linewidth=2)
    plt.plot(val_losses,   label='val',   color='#fd79a8', linewidth=2, linestyle='--')
    plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
    plt.title('Seq2Seq Transformer — Training Loss (comtrans en→fr)')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Show translations for a handful of validation examples
print('\n── Sample Translations ─────────────────────────────────────────')
indices = list(range(0, min(10, len(en_sents)), 2))
for i in indices:
    en = en_sents[i]
    ref = fr_sents[i]
    hyp = greedy_decode(model, en)
    print(f'EN  : {" ".join(en)}')
    print(f'REF : {" ".join(ref)}')
    print(f'PRED: {" ".join(hyp)}')
    print()

## Step 12: Visualise Cross-Attention Weights

One of the original motivations for the transformer was **interpretability**:
the cross-attention weights directly show which source words the model
attends to when generating each target word.

In [ ]:
def get_cross_attention(model, src_words, tgt_words):
    """Return cross-attention weights (H, T, S) from the first decoder layer."""
    model.eval()
    with torch.no_grad():
        src_ids = [SOS_IDX] + [en_vocab.get(w.lower(), UNK_IDX) for w in src_words[:MAX_LEN]] + [EOS_IDX]
        tgt_ids = [SOS_IDX] + [fr_vocab.get(w.lower(), UNK_IDX) for w in tgt_words[:MAX_LEN]]
        src_t = torch.tensor([src_ids], dtype=torch.long, device=device)
        tgt_t = torch.tensor([tgt_ids], dtype=torch.long, device=device)

        memory = model.encode(src_t)                                  # (1, S, d_model)
        T = tgt_t.size(1)
        tgt_mask = make_causal_mask(T, device)
        x = model.tgt_pos_enc(model.tgt_embedding(tgt_t))
        x = model.decoder_layers[0].norm1(
                x + model.decoder_layers[0].drop(
                    model.decoder_layers[0].self_attn(x, x, x, mask=tgt_mask)[0]
                ))
        _, cross_w = model.decoder_layers[0].cross_attn(x, memory, memory)  # (1,H,T,S)
    return cross_w[0].cpu()  # (H, T, S)

idx = 0
cross_w = get_cross_attention(model, en_sents[idx], fr_sents[idx])
H = cross_w.size(0)
S_words = ['<SOS>'] + en_sents[idx] + ['<EOS>']
T_words = ['<SOS>'] + fr_sents[idx]

if HAS_MPL:
    fig, axes = plt.subplots(1, min(H, 4), figsize=(14, 4))
    if H == 1: axes = [axes]
    for h, ax in enumerate(axes):
        mat = cross_w[h, :len(T_words), :len(S_words)].numpy()
        ax.imshow(mat, cmap='Blues', aspect='auto')
        ax.set_xticks(range(len(S_words))); ax.set_xticklabels(S_words, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(T_words))); ax.set_yticklabels(T_words, fontsize=7)
        ax.set_title(f'Head {h+1}', fontsize=9)
    plt.suptitle('Cross-Attention (row=target, col=source)', fontsize=11)
    plt.tight_layout(); plt.show()

## Step 13: Encoder-Decoder Architecture Diagram

In [ ]:
if HAS_MPL:
    fig, (ax_enc, ax_dec) = plt.subplots(1, 2, figsize=(13, 10))

    def draw_stack(ax, title, blocks, color_map, show_residuals=True):
        ax.set_xlim(0, 8); ax.set_ylim(0, len(blocks) * 1.5 + 1); ax.axis('off')
        ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
        for i, (label, color) in enumerate(blocks):
            y = i * 1.5 + 0.5
            rect = mpatches.FancyBboxPatch((1, y), 6, 1.1,
                boxstyle='round,pad=0.05', facecolor=color, edgecolor='#2d3436', lw=1.5)
            ax.add_patch(rect)
            ax.text(4, y + 0.55, label, ha='center', va='center',
                    fontsize=8, fontweight='bold')
            if i > 0:
                ax.annotate('', xy=(4, y), xytext=(4, (i-1)*1.5+1.6),
                    arrowprops=dict(arrowstyle='->', color='#636e72', lw=1.5))

    enc_blocks = [
        ('(B, S) Source Tokens',            '#dfe6e9'),
        ('Embedding + Positional Enc\n(B, S, d_model)', '#74b9ff'),
        ('Multi-Head Self-Attention\n(B, S, d_model)',  '#a29bfe'),
        ('Add & LayerNorm',                 '#ffeaa7'),
        ('Feed-Forward\n(B, S, d_model)',   '#55efc4'),
        ('Add & LayerNorm',                 '#ffeaa7'),
        ('memory  (B, S, d_model)',         '#fd79a8'),
    ]
    dec_blocks = [
        ('(B, T) Target Tokens (teacher-forced)', '#dfe6e9'),
        ('Embedding + Positional Enc\n(B, T, d_model)',     '#74b9ff'),
        ('Masked Self-Attention (causal)\n(B, T, d_model)', '#a29bfe'),
        ('Add & LayerNorm',                                 '#ffeaa7'),
        ('Cross-Attention\nQ←decoder  K,V←memory\n(B, T, d_model)', '#fab1a0'),
        ('Add & LayerNorm',                                 '#ffeaa7'),
        ('Feed-Forward\n(B, T, d_model)',                   '#55efc4'),
        ('Add & LayerNorm',                                 '#ffeaa7'),
        ('Linear → Logits\n(B, T, tgt_vocab_size)',         '#fd79a8'),
    ]
    draw_stack(ax_enc, 'Encoder', enc_blocks, {})
    draw_stack(ax_dec, 'Decoder', dec_blocks, {})

    ax_dec.annotate('memory\n(from encoder)', xy=(1.0, 4 * 1.5 + 1.05),
        xytext=(-0.2, 4 * 1.5 + 1.05),
        fontsize=7.5, color='#e17055',
        arrowprops=dict(arrowstyle='->', color='#e17055', lw=1.5))

    ax_enc.text(4, len(enc_blocks)*1.5+0.6, '× N encoder layers',
        ha='center', fontsize=8, color='#636e72', style='italic')
    ax_dec.text(4, len(dec_blocks)*1.5+0.6, '× N decoder layers',
        ha='center', fontsize=8, color='#636e72', style='italic')

    plt.suptitle('Encoder-Decoder Transformer — Machine Translation', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

## Final Concept Check: Encoder-Decoder Transformers

**Q1.** During teacher-forced training the decoder input is `tgt[:, :-1]`
and the target is `tgt[:, 1:]`.  Write out what both look like for the
French sentence `<SOS> le chat dort <EOS>`.  What is the model learning
at position 0 of the output?

**Q2.** In cross-attention the Q has shape `(B, H, T, d_k)` and K has
shape `(B, H, S, d_k)`.  The score matrix is `Q @ K^T`.  What are the
shapes of the score matrix and the output?  What would go wrong if you
accidentally passed `memory` as Q and `tgt` as K/V?

**Q3.** Explain the **exposure bias** problem.  Why does the gap between
teacher-forced training and autoregressive inference matter more for long
sequences than short ones?

**Q4.** T5 (Text-to-Text Transfer Transformer) frames every NLP task as
sequence-to-sequence.  Give two examples of tasks that are *not* obviously
seq2seq but can be cast that way, and describe what the source and target
sequences would be.

**Q5.** Gradient clipping (`clip_grad_norm_(..., 1.0)`) was added to the
training loop.  Why is it particularly important for encoder-decoder models
that are deeper than encoder-only models?

In [ ]:
# Q1:
# Q2:
# Q3:
# Q4:
# Q5: